In [1]:
import pandas as pd
import numpy as np

TARGETS = ["X", "Y"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3", 
    "LSG_1",
    "LSG_2"  
]

results = pd.read_excel("resultados.xlsx")


In [2]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [3]:
df_summary_top10

,dataset,target,mean_r2,std_r2,mean_mse,std_mse
0,Train_1,X,0.996054,0.000264,0.000421,0.000028
1,Train_2,X,0.998036,0.000204,0.000165,0.000017
2,Val,X,0.852879,0.009116,0.013953,0.000865
3,Test_1,X,0.965230,0.012249,0.003608,0.001271
4,Test_2,X,0.990551,0.002770,0.000775,0.000227
5,Test_3,X,0.947524,0.008288,0.004794,0.000757
6,Train_1,Y,0.963517,0.007450,0.003045,0.000622
7,Train_2,Y,0.968466,0.003779,0.003367,0.000403
8,Val,Y,0.962799,0.005425,0.002762,0.000403
9,Test_1,Y,0.986697,0.003194,0.001085,0.000261


In [4]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
   Target  Dataset                          Best_Model         Neurons  \
0       X  Train_1     model_arch128-64_r0.01_seed1586       [128, 64]   
1       Y  Train_1         model_arch8-4_r0.9_seed8857          [8, 4]   
2       X  Train_2    model_arch32-16-8_r0.01_seed1586     [32, 16, 8]   
3       Y  Train_2        model_arch264_r0.01_seed1586           [264]   
4       X      Val      model_arch32-16_r0.01_seed8478        [32, 16]   
5       Y      Val       model_arch16-8_r0.01_seed1586         [16, 8]   
6       X   Test_1       model_arch16-8_r0.01_seed1658         [16, 8]   
7       Y   Test_1         model_arch32_r0.01_seed8478            [32]   
8       X   Test_2       model_arch32-16_r0.9_seed1586        [32, 16]   
9       Y   Test_2          model_arch16_r0.9_seed8857            [16]   
10      X   Test_3  model_arch264-128-64_r0.9_seed1658  [264, 128, 64]   
11      Y   Test_3       model_arch16-8_r0.01_seed8857         [16